### Import dependencies and load ConnectionConfig

In [1]:
from pyspark.sql.functions import col, struct, collect_list
import sys
import os
# Add the current directory to sys.path to ensure ConnectionConfig is found
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import ConnectionConfig as cc
from importlib import reload

In [2]:
reload(cc)

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

/home/jovyan/work/04-mongodb-nosql/ConnectionConfig.py
Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/spark

### Select connection profile and create JDBC config

In [3]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

jdbc:postgresql://postgres-data4:5432/catchem?user=postgres&password=data&ssl=false


### Start local Spark cluster

In [4]:
spark = cc.startLocalCluster("mongoExport")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


### Load operational tables from PostgreSQL

In [5]:
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_country = cc.read_postgres_without_partition(spark, "country")
df_operational_stage = cc.read_postgres_without_partition(spark, "stage")
df_operational_treasure_stages = cc.read_postgres_without_partition(spark, "treasure_stages")

### Convert binary IDs to hex strings

In [6]:
from pyspark.sql.functions import col, hex

df_operational_treasure = df_operational_treasure.withColumn("hex_id", hex(col("id")))
# Convert binary IDs to readable hex strings
# Handled at the end of transformation

## Transform and nest data (location, logs, stages)

In [7]:
# 1. Enrich Treasure with City and Country info
treasures_with_location = df_operational_treasure.join(
    df_operational_city,
    df_operational_treasure.city_city_id == df_operational_city.city_id,
    "left"
).join(
    df_operational_country,
    df_operational_city.country_code == df_operational_country.code,
    "left"
).select(
    df_operational_treasure["*"],
    df_operational_city.city_name,
    df_operational_city.latitude.alias("city_lat"), # Y
    df_operational_city.longitude.alias("city_long"), # X
    df_operational_city.postal_code,
    df_operational_country.name.alias("country_name")
)

# 3. Aggregate Stages into a list for each treasure
nested_stages = df_operational_treasure_stages.join(
    df_operational_stage,
    df_operational_treasure_stages.stages_id == df_operational_stage.id,
    "left"
).select(
    df_operational_treasure_stages.treasure_id,
    struct(
        df_operational_stage["id"].alias("id"),
        col("description"),
        col("latitude"),
        col("longitude"),
        col("sequence_number"),
        col("type"),
        col("visibility"),
        col("container_size")
    ).alias("stage_details")
).groupBy("treasure_id").agg(collect_list("stage_details").alias("stages"))

# 4. Final Join and Nesting
final_treasures_df = treasures_with_location.join(
    nested_stages,
    treasures_with_location.id == nested_stages.treasure_id,
    "left"
).select(
    col("id").alias("id"),
    col("hex_id").alias("hex_id"),
    col("difficulty"),
    col("terrain"),
    struct(
        col("city_name").alias("city_name"),
        col("country_name").alias("country_name"),
        col("postal_code").alias("postal_code"),
        struct(
            col("city_lat").alias("latitude"),
            col("city_long").alias("longitude")
        ).alias("coordinates")
    ).alias("location"),
    col("stages")
)



### Preview nested structure

In [8]:
final_treasures_df.show(1, truncate=True)

+--------------------+--------------------+----------+-------+--------------------+--------------------+
|                  id|              hex_id|difficulty|terrain|            location|              stages|
+--------------------+--------------------+----------+-------+--------------------+--------------------+
|[00 03 72 3C 7C C...|0003723C7CC34262A...|         3|      2|{Sarandapalli, In...|[{[1B 32 E8 9A A6...|
+--------------------+--------------------+----------+-------+--------------------+--------------------+
only showing top 1 row


### Save the first row as a JSON file to verify the structure before exporting the entire dataset

In [9]:
import json

# Extract first row as JSON string
row_json = final_treasures_df.limit(1).toJSON().collect()[0]

# Choose output path
single_row_path = "treasures_export/first_row.json"

# Write JSON to file
with open(single_row_path, "w") as f:
    f.write(row_json)

print(f"First row saved to: {single_row_path}")


First row saved to: /home/jovyan/work/04-mongodb-nosql/treasures_export/first_row.json


## Export to JSON for mongoimport

In [29]:
import shutil
import glob

# Paths
base_export_path = "treasures_export"
temp_output_dir = os.path.join(base_export_path, "treasure_output")
final_json_file = os.path.join(base_export_path, "treasure.json")

# Clean up previous runs
if os.path.exists(temp_output_dir):
    shutil.rmtree(temp_output_dir)
if os.path.exists(final_json_file):
    os.remove(final_json_file)

# Efficiently export using Spark's native JSON writer
# coalesce(1) ensures a single part-file is created
final_treasures_df.coalesce(1).write.json(temp_output_dir)

# Find the generated part file and move it to the parent directory as 'treasure.json'
part_file = glob.glob(os.path.join(temp_output_dir, "part-*.json"))[0]
shutil.copy(part_file, final_json_file)

# Cleanup: delete the temporary Spark output directory
shutil.rmtree(temp_output_dir)

print(f"Data exported successfully to: {final_json_file}")

Data exported successfully to: /home/jovyan/work/04-mongodb-nosql/treasures_export/treasure.json


#### Reset Jupyter state

In [10]:
%reset